In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

raw = Path("../data/raw")
clean = Path("../data/clean")
clean.mkdir(parents=True, exist_ok=True)

sessions = pd.read_parquet(raw / "sessions.parquet")
events = pd.read_parquet(raw / "events.parquet")
orders = pd.read_parquet(raw / "orders.parquet")
users = pd.read_parquet(raw / "users.parquet")

pd.set_option("display.float_format","{:,.2f}".format)
pd.set_option("display.max_columns", 50)

for name, df in[("sessions",sessions), ("events", events), ("orders", orders), ("users", users)]:
    print(name, df.shape)

sessions (903653, 17)
events (1059291, 10)
orders (11552, 11)
users (714167, 8)


In [2]:
# missing values of sessions
nulls = sessions.isna().sum().to_frame("nulls")
nulls["pct"] = (nulls["nulls"] / len(sessions) * 100).round(3)
nulls[nulls["nulls"] > 0]

,nulls,pct
pageviews,100,0.01
transactions,892101,98.72
session_revenue_usd,892138,98.73


In [3]:
# value ranges
cols = ["visit_number","pageviews","total_hits","session_revenue_usd"]
print(sessions[cols].describe(percentiles=[.5,.9,.99,.999]))
print("date range:", sessions["session_date"].min(),"to",sessions["session_date"].max())
print("visit_number < 1:", (sessions["visit_number"] < 1).sum())
print("negative pageviews:", (sessions["pageviews"] < 0).sum())
print("negative total_hits:",(sessions["total_hits"] < 0).sum())

       visit_number  pageviews  total_hits  session_revenue_usd
count    903,653.00 903,553.00  903,653.00            11,515.00
mean           2.26       3.85        4.60               133.74
std            9.28       7.03        9.64               448.29
min            1.00       1.00        1.00                 0.01
50%            1.00       1.00        2.00                49.45
90%            3.00       9.00       11.00               279.05
99%           20.00      33.00       44.00             1,269.77
99.9%        150.00      73.00      105.00             5,163.69
max          395.00     469.00      500.00            23,129.50
date range: 2016-08-01 00:00:00 to 2017-08-01 00:00:00
visit_number < 1: 0
negative pageviews: 0
negative total_hits: 0


In [4]:
#hits versus pageviews  
print("hits < pageviews:",(sessions["total_hits"] < sessions["pageviews"]).sum())

hits < pageviews: 0


In [5]:
#bounce consistency
print("bounce with >1 pageview:", (sessions["is_bounce"] & (sessions["pageviews"] > 1)).sum())

bounce with >1 pageview: 0


In [6]:
#purchase consistency
has_tx = sessions["transactions"].notna()
has_rev = sessions["session_revenue_usd"].notna()
pd.crosstab(has_tx, has_rev, rownames=["has_transactions"], colnames=["has_revenue"])

has_revenue,False,True
has_transactions,,
False,892101,0
True,37,11515


In [7]:
#category values
for c in ["channel_grouping","device_category"]:
    print(sessions[c].value_counts(dropna=False),"\n")
print(sessions["country"].value_counts(dropna=False).head(15))
print("distinct countries:", sessions["country"].nunique())

channel_grouping
Organic Search    381561
Social            226117
Direct            143026
Referral          104838
Paid Search        25326
Affiliates         16403
Display             6262
(Other)              120
Name: count, dtype: int64 

device_category
desktop    664479
mobile     208725
tablet      30449
Name: count, dtype: int64 

country
United States     364744
India              51140
United Kingdom     37393
Canada             25869
Vietnam            24598
Turkey             20522
Thailand           20123
Germany            19980
Brazil             19783
Japan              19731
France             15832
Mexico             13225
Taiwan             12996
Australia          12698
Russia             11662
Name: count, dtype: int64
distinct countries: 222


In [8]:
#extereme sessions
p999 = sessions["total_hits"].quantile(.999)
print("99.9th percentile:", p999, " | max:",sessions["total_hits"].max())
print("sessions above the 99.9th percentile:", (sessions["total_hits"] > p999).sum())
sessions.nlargest(10, "total_hits")[
    ["session_key","total_hits","pageviews","channel_grouping","device_category","country"]
]

99.9th percentile: 105.0  | max: 500
sessions above the 99.9th percentile: 891


,session_key,total_hits,pageviews,channel_grouping,device_category,country
38910,4903244865428862983-1471399014-20160816,500,333,Direct,desktop,United States
58172,741993131378886687-1472061509-20160824,500,327,Organic Search,desktop,United States
60710,7737049103084491745-1472172400-20160825,500,351,Organic Search,desktop,Venezuela
60711,3924372865099736100-1472132217-20160825,500,431,Organic Search,desktop,Venezuela
63070,743123551680199202-1472216705-20160826,500,469,Social,desktop,Venezuela
124168,2001696391343143037-1474483351-20160921,500,300,Direct,desktop,United States
254998,0270832659599010298-1478229181-20161103,500,341,Social,desktop,Japan
476608,1106901861011569877-1484949128-20170120,500,324,Organic Search,desktop,United States
571850,5526675926038480325-1488663704-20170304,500,466,Direct,desktop,United States
602530,6879336706336070910-1489823400-20170318,500,400,Referral,desktop,Venezuela


In [9]:
#split-visit flags
print("split_rows:", sessions["is_split_visit"].sum(),
      " | continuous_rows:", sessions["is_continuation_row"].sum())
print(sessions[sessions["is_split_visit"]].groupby("session_id").size().value_counts())

split_rows: 1796  | continuous_rows: 898
2    898
Name: count, dtype: int64


In [10]:
# missing values of events
nulls = events.isna().sum().to_frame("nulls")
nulls["pct"] = (nulls["nulls"] / len(sessions) * 100).round(3)
nulls

,nulls,pct
event_id,0,0.00
session_key,0,0.00
session_id,0,0.00
full_visitor_id,0,0.00
event_ts,0,0.00
hit_number,0,0.00
hit_type,0,0.00
action_type,0,0.00
event_name,0,0.00
transaction_id,1034759,114.51


In [11]:
#timing
m = events.merge(sessions[["session_key","visit_start_ts"]], on = "session_key", how="left")
offset_h = (m["event_ts"] - m["visit_start_ts"]).dt.total_seconds() / 3600
print("events before session start:", (offset_h < 0).sum())
print("events more than 24h after session start:", (offset_h > 24).sum())
print("max offset (hours):", offset_h.max())
print("event_ts range:", events["event_ts"].min(), "to", events["event_ts"].max())
del m, offset_h

events before session start: 0
events more than 24h after session start: 0
max offset (hours): 4.479332777777778
event_ts range: 2016-08-01 07:06:52.637000+00:00 to 2017-08-02 06:58:13.655000+00:00


In [12]:
#events per session
per_session = events.groupby("session_key").size()
print(per_session.describe(percentiles=[.5,.9,.99,.999]))

count   133,343.00
mean          7.94
std          10.62
min           1.00
50%           5.00
90%          17.00
99%          47.00
99.9%       108.00
max         463.00
dtype: float64


In [13]:
#skipped funnel steps
stage_map = {"product_view": "view", "add_to_cart": "cart",
             "checkout": "checkout", "purchase": "purchase"}
st = events[events["event_name"].isin(stage_map)].copy()
st["stage"] = st["event_name"].map(stage_map)
first = st.groupby(["session_key", "stage"])["event_ts"].min().unstack()
has = first.notna()

print("sessions reaching each stage:\n", has.sum(), "\n")
p = has["purchase"]
print("purchase without cart:    ", (p & ~has["cart"]).sum())
print("purchase without view:    ", (p & ~has["view"]).sum())
print("checkout without cart:    ", (has["checkout"] & ~has["cart"]).sum())
print("cart without view:        ", (has["cart"] & ~has["view"]).sum())
print("purchase before cart:     ", (first["purchase"] < first["cart"]).sum())
print("cart before first view:   ", (first["cart"] < first["view"]).sum())

sessions reaching each stage:
 stage
cart         50139
checkout     22407
purchase     11552
view        124188
dtype: int64 

purchase without cart:     1440
purchase without view:     2179
checkout without cart:     3272
cart without view:         5612
purchase before cart:      43
cart before first view:    1998


In [14]:
#missing values of orders
nullls = orders.isna().sum().to_frame("nulls")
nulls["pct"] = (nulls["nulls"] / len(orders) * 100).round(3)
nulls[nulls["nulls"] > 0]

,nulls,pct
transaction_id,1034759,"8,957.40"


In [15]:
#revenue distribution
rev = orders["revenue_usd"]
print(rev.describe(percentiles = [.01,.1, .5, .9, .99, .999]))
print("median:", round(rev.median(), 2), " | mean:", round(rev.mean(), 2))
print("revenue < 1:", (rev < 1).sum(), " | revenue < 5", (rev < 5).sum())

count   11,515.00
mean       141.33
std        492.32
min          1.20
1%           7.98
10%         19.59
50%         54.28
90%        285.77
99%      1,268.36
99.9%    5,553.20
max     23,952.56
Name: revenue_usd, dtype: float64
median: 54.28  | mean: 141.33
revenue < 1: 0  | revenue < 5 46


In [16]:
#outliers and concentration
p999_rev = rev.quantile(.999)
print("99.9th percentile:", round(p999_rev, 2), " | orders above it:", (rev > p999_rev).sum())
print("top 10 orders share of revenue: %.1f%%" % (rev.nlargest(10).sum() / rev.sum() * 100))
print("top 1%% of orders share of revenue: %.1f%%" % 
      (rev.nlargest(int(len(orders) * 0.01)).sum() / rev.sum() * 100))
orders.nlargest(15, "revenue_usd").merge(
    sessions[["session_key","total_hits","channel_grouping","country"]], on="session_key"
)[["transaction_id","order_date","revenue_usd","tax_usd","shipping_usd",
  "total_hits","channel_grouping","country"]]

99.9th percentile: 5553.2  | orders above it: 12
top 10 orders share of revenue: 8.0%
top 1% of orders share of revenue: 21.7%


,transaction_id,order_date,revenue_usd,tax_usd,shipping_usd,total_hits,channel_grouping,country
0,ORD201704052324,2017-04-05,"23,952.56",NaN,13.50,58,Display,United States
1,ORD201702142258,2017-02-14,"17,859.50",NaN,10.50,91,Direct,United States
2,ORD201609161805,2016-09-16,"16,032.75",NaN,15.50,136,Direct,United States
3,ORD201704182260,2017-04-18,"15,634.61",NaN,9.00,77,Display,United States
4,ORD201707182786,2017-07-18,"14,660.12",NaN,12.00,41,Direct,United States
5,ORD201707132695,2017-07-13,"11,511.81",NaN,13.00,156,Direct,United States
6,ORD201703242139,2017-03-24,"8,680.83",NaN,9.50,50,Display,United States
7,ORD201704272630,2017-04-27,"8,257.80",763.14,139.46,32,Referral,United States
8,ORD201612123195,2016-12-12,"7,003.50",691.89,837.74,15,Referral,United States
9,ORD201704072079,2017-04-07,"6,831.96",NaN,11.50,227,Display,United States


In [17]:
#gap with session-level revenue
g = orders.merge(sessions[["session_key","session_revenue_usd"]], on="session_key", how="left")
g["diff"] = g["revenue_usd"] - g["session_revenue_usd"]
print(g["diff"].describe(percentiles=[.01, .05, .5, .95, .99]))
print("|diff| > 1:", (g["diff"].abs() > 1).sum(), "| |diff| > 100:", (g["diff"].abs() > 100).sum())
print("both revenues null:", (g["revenue_usd"].isna() & g["session_revenue_usd"].isna()).sum())
print("total gap:", round(g["diff"].sum(), 2))
big = g["diff"].abs().nlargest(14).index
print("gap without the 14 largest differences:", round(g.drop(big)["diff"].sum(), 2))

ship = g["shipping_usd"].fillna(0)
tax = g["tax_usd"].fillna(0)
print("diff == shipping:", ((g["diff"] - ship).abs() < 0.01).sum())
print("diff == shipping + tax:", ((g["diff"] - ship - tax).abs() < 0.01).sum())

count   11,515.00
mean         7.59
std        140.21
min     -1,958.30
1%          -1.20
5%           1.00
50%          5.00
95%          9.00
99%         28.99
max     10,020.17
Name: diff, dtype: float64
|diff| > 1: 10303 | |diff| > 100: 97
both revenues null: 37
total gap: 87358.3
gap without the 14 largest differences: 64074.65
diff == shipping: 1
diff == shipping + tax: 1


In [18]:
#tax and shipping distribution
for c in ["tax_usd", "shipping_usd"]:
    print(c, "nulls", orders[c].isna().sum(), "| negatives:", (orders[c] < 0).sum(),
          "| larger than revenue:", (orders[c] > orders["revenue_usd"]).sum())
print(orders[["tax_usd","shipping_usd"]].describe())

tax_usd nulls 3469 | negatives: 0 | larger than revenue: 8
shipping_usd nulls 72 | negatives: 0 | larger than revenue: 373
       tax_usd  shipping_usd
count 8,083.00     11,480.00
mean     15.82         18.83
std      24.24         33.95
min       1.26          7.00
25%       7.17          9.50
50%      10.35         12.50
75%      15.11         15.50
max     763.14        960.74


In [19]:
#Date embedded in the order ID versus order_date
id_date = pd.to_datetime(orders["transaction_id"].str.extract(r"ORD(\d{8})")[0],
                         format="%Y%m%d", errors="coerce")
print("IDs without a valid date:",id_date.isna().sum())
d = (id_date - orders["order_date"]).dt.days
print(d.value_counts().head())

IDs without a valid date: 0
0    11552
Name: count, dtype: int64


In [20]:
#links to sessions
linked = orders.merge(sessions[["session_key","transactions"]], on="session_key", how="left")
print("orders with no matching session:", linked["transactions"].isna().sum())
print("orders per full_visitor_id (top):",orders["full_visitor_id"].value_counts().head(3).to_dict())

orders with no matching session: 0
orders per full_visitor_id (top): {'7813149961404844386': 33, '4984366501121503466': 16, '1957458976293878100': 16}


In [21]:
by_v = orders.groupby("full_visitor_id").agg(
    orders=("transaction_id", "size"),
    revenue=("revenue_usd", "sum"),
).sort_values("orders", ascending=False)
print(by_v.head(5))
print("visitors with 2+ orders:", (by_v["orders"] > 1).sum(),
      "| of", len(by_v), "buying visitors")
print("orders from the top 5 visitors:", by_v["orders"].head(5).sum())

                     orders   revenue
full_visitor_id                      
7813149961404844386      33  2,689.34
1957458976293878100      16 88,395.15
4984366501121503466      16  9,002.95
6760732402251466726      15  4,518.96
2402527199731150932      15  3,374.50
visitors with 2+ orders: 971 | of 10022 buying visitors
orders from the top 5 visitors: 95


In [22]:
#missing values of users
nulls = users.isna().sum().to_frame("nulls")
nulls["pct"] = (nulls["nulls"] / len(users) * 100).round(3)
nulls[nulls["nulls"] > 0]

,nulls,pct


In [23]:
#first session date = users["first_session_date"]
first_by_sessions = sessions.groupby("full_visitor_id")["session_date"].min()
chk = users.set_index("full_visitor_id")["first_session_date"]
print("visitors with different first date:", (chk != first_by_sessions.reindex(chk.index)).sum())
print("is_true_new inconsistent with first_visit_number:",
      (users["is_true_new"] != (users["first_visit_number"] == 1)).sum())

visitors with different first date: 0
is_true_new inconsistent with first_visit_number: 0


In [24]:
#123 visitors with out-of-order visit numbers
has_one = sessions.loc[sessions["visit_number"] == 1, "full_visitor_id"].unique()
users["has_inconsistent_visit_number"] = (
    users["full_visitor_id"].isin(has_one) & (users["first_visit_number"] > 1)
)
print("inconsistent visit numbers:", users["has_inconsistent_visit_number"].sum())

inconsistent visit numbers: 123


In [25]:
# Which visitors enter cohort analysis 
users["in_cohort_analysis"] = users["is_true_new"] & (users["cohort_month"] < "2017-08-01")
print("in cohort analysis:", users["in_cohort_analysis"].sum())
print(users["cohort_month"].dt.to_period("M").value_counts().sort_index())

in cohort analysis: 699633
cohort_month
2016-08    61699
2016-09    56481
2016-10    81488
2016-11    95764
2016-12    59378
2017-01    49572
2017-02    48310
2017-03    54709
2017-04    52424
2017-05    48656
2017-06    48673
2017-07    55118
2017-08     1895
Freq: M, Name: count, dtype: int64


In [26]:
# Heavy visitors
spv = sessions.groupby("full_visitor_id").size()
print(spv.describe(percentiles=[.5, .9, .99, .999]))
heavy_cut = spv.quantile(0.999)
users["is_heavy_visitor"] = users["full_visitor_id"].map(spv) > heavy_cut
print("99.9th percentile of sessions per visitor:", heavy_cut,
      "| heavy visitors:", users["is_heavy_visitor"].sum())
print(spv.nlargest(10))

count   714,167.00
mean          1.27
std           1.43
min           1.00
50%           1.00
90%           2.00
99%           5.00
99.9%        15.00
max         278.00
dtype: float64
99.9th percentile of sessions per visitor: 15.0 | heavy visitors: 635
full_visitor_id
1957458976293878100    278
0824839726118485274    255
3608475193341679870    201
1856749147915772585    199
3269834865385146569    155
0720311197761340948    153
7634897085866546110    148
4038076683036146727    138
0232377434237234751    135
3694234028523165868    129
dtype: int64


In [27]:
# sessions flag
p999_hits = sessions["total_hits"].quantile(0.999)
has_tx = sessions["transactions"].notna()
has_rev = sessions["session_revenue_usd"].notna()
sessions["is_hit_cap_session"] = sessions["total_hits"] == 500
sessions["is_extreme_session"] = sessions["total_hits"] > p999_hits
sessions["has_revenue_gap"] = has_tx & ~has_rev
sessions["is_valid_purchase_row"] = has_tx & has_rev

In [28]:
# orders flag
p999_rev = orders["revenue_usd"].quantile(0.999)
orders["is_valid_order"] = orders["revenue_usd"].notna() & (orders["revenue_usd"] > 0)
orders["is_large_order_outlier"] = orders["revenue_usd"] > p999_rev

In [29]:
# ---- users flags ----
has_one = sessions.loc[sessions["visit_number"] == 1, "full_visitor_id"].unique()
users["has_inconsistent_visit_number"] = (
    users["full_visitor_id"].isin(has_one) & (users["first_visit_number"] > 1)
)
users["in_cohort_analysis"] = users["is_true_new"] & (users["cohort_month"] < "2017-08-01")

spv = sessions.groupby("full_visitor_id").size()
users["is_heavy_visitor"] = users["full_visitor_id"].map(spv) > spv.quantile(0.999)

vo = orders[orders["is_valid_order"]].groupby("full_visitor_id").size()
users["valid_orders"]   = users["full_visitor_id"].map(vo).fillna(0).astype(int)
users["is_heavy_buyer"] = users["valid_orders"] >= 10

In [30]:
# ---- save the clean files ----
for name, df in [("sessions", sessions), ("events", events), ("orders", orders), ("users", users)]:
    df.to_parquet(clean / f"{name}.parquet", index=False)
    print("saved", name, df.shape)

saved sessions (903653, 21)
saved events (1059291, 10)
saved orders (11552, 13)
saved users (714167, 13)


In [31]:
#checking the clean files against the raw files
chk = {}
for name in ["sessions", "events", "orders", "users"]:
    r = pd.read_parquet(raw / f"{name}.parquet")
    c = pd.read_parquet(clean / f"{name}.parquet")
    new_cols = [col for col in c.columns if col not in r.columns]
    same_rows = len(r) == len(c)
    same_values = c[r.columns].equals(r)          # original columns untouched
    print(f"{name}: raw {len(r):,} rows, clean {len(c):,} rows | same rows: {same_rows}"
          f" | original columns unchanged: {same_values} | new columns: {new_cols}")

s = pd.read_parquet(clean / "sessions.parquet")
o = pd.read_parquet(clean / "orders.parquet")
u = pd.read_parquet(clean / "users.parquet")
print("\nflag counts (expected in brackets)")
print("is_hit_cap_session:", s["is_hit_cap_session"].sum(), "[10]")
print("is_extreme_session:", s["is_extreme_session"].sum(), "[891]")
print("has_revenue_gap:", s["has_revenue_gap"].sum(), "[37]")
print("is_valid_purchase_row:", s["is_valid_purchase_row"].sum(), "[11,515]")
print("is_valid_order:", o["is_valid_order"].sum(), "[11,515]")
print("is_large_order_outlier:", o["is_large_order_outlier"].sum(), "[12]")
print("has_inconsistent_visit_number:", u["has_inconsistent_visit_number"].sum(), "[123]")
print("in_cohort_analysis:", u["in_cohort_analysis"].sum(), "[699,633]")
print("is_heavy_visitor:", u["is_heavy_visitor"].sum(), "[635]")
print("is_heavy_buyer:", u["is_heavy_buyer"].sum(), "[not yet known: record it]")

sessions: raw 903,653 rows, clean 903,653 rows | same rows: True | original columns unchanged: True | new columns: ['is_hit_cap_session', 'is_extreme_session', 'has_revenue_gap', 'is_valid_purchase_row']
events: raw 1,059,291 rows, clean 1,059,291 rows | same rows: True | original columns unchanged: True | new columns: []
orders: raw 11,552 rows, clean 11,552 rows | same rows: True | original columns unchanged: True | new columns: ['is_valid_order', 'is_large_order_outlier']
users: raw 714,167 rows, clean 714,167 rows | same rows: True | original columns unchanged: True | new columns: ['has_inconsistent_visit_number', 'in_cohort_analysis', 'is_heavy_visitor', 'valid_orders', 'is_heavy_buyer']

flag counts (expected in brackets)
is_hit_cap_session: 10 [10]
is_extreme_session: 891 [891]
has_revenue_gap: 37 [37]
is_valid_purchase_row: 11515 [11,515]
is_valid_order: 11515 [11,515]
is_large_order_outlier: 12 [12]
has_inconsistent_visit_number: 123 [123]
in_cohort_analysis: 699633 [699,633]


In [32]:
#data dictionary
def dictionary_skeleton(name, df):
    rows = []
    for col in df.columns:
        ex = df[col].dropna().iloc[0] if df[col].notna().any() else ""
        rows.append(f"| `{col}` | {df[col].dtype} | | | | {int(df[col].isna().sum()):,} | {str(ex)[:40]} |")
    return (f"\n## {name}: one row per ____ (grain)\n"
            "| Column | Type | Meaning | Source field | Caveat | Nulls | Example |\n"
            "|---|---|---|---|---|---|---|\n" + "\n".join(rows) + "\n")

text = "# Data dictionary (clean tables)\n"
for name, df in [("sessions", s), ("events", pd.read_parquet(clean / "events.parquet")),
                 ("orders", o), ("users", u)]:
    text += dictionary_skeleton(name, df)

Path("../docs/data_dictionary.md").write_text(text, encoding="utf-8")

3696

In [33]:
orders.columns.tolist()

['transaction_id',
 'session_id',
 'session_key',
 'full_visitor_id',
 'order_date',
 'order_ts',
 'hit_number',
 'revenue_usd',
 'tax_usd',
 'shipping_usd',
 'currency',
 'is_valid_order',
 'is_large_order_outlier']